In [4]:
# Cell 1: imports & helper functions
import json
import numpy as np
import pandas as pd
from itertools import combinations
from typing import List, Tuple, Dict

from scipy.stats import ks_2samp, pearsonr
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from tqdm import tqdm

from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

# ---------- small helpers ----------
def total_variation_distance_counts(real_counts: np.ndarray, synth_counts: np.ndarray) -> float:
    p = real_counts.astype(float) / (real_counts.sum() + 1e-12)
    q = synth_counts.astype(float) / (synth_counts.sum() + 1e-12)
    return float(np.abs(p - q).sum())

# ---------- column fidelity ----------
def column_fidelity(real_df: pd.DataFrame, synth_df: pd.DataFrame,
                    categorical_cols: List[str], numeric_cols: List[str]) -> Tuple[Dict, float]:
    per_col = {}
    for col in numeric_cols:
        a = real_df[col].dropna().values
        b = synth_df[col].dropna().values
        if len(a) == 0 or len(b) == 0:
            ks_stat = 1.0
        else:
            ks_stat = ks_2samp(a, b).statistic
        omega = 1.0 - float(ks_stat)
        per_col[col] = {"type": "numeric", "ks": float(ks_stat), "omega_col": float(omega)}

    for col in categorical_cols:
        a = real_df[col].astype(str).fillna("__nan__").values
        b = synth_df[col].astype(str).fillna("__nan__").values
        real_counts = pd.Series(a).value_counts().sort_index()
        synth_counts = pd.Series(b).value_counts().reindex(real_counts.index, fill_value=0)
        tvd = total_variation_distance_counts(real_counts.values, synth_counts.values)
        omega = 1.0 - 0.5 * float(tvd)
        per_col[col] = {"type": "categorical", "tvd": float(tvd), "omega_col": float(omega)}

    Omega_col = float(np.mean([v["omega_col"] for v in per_col.values()]))
    return per_col, Omega_col

# ---------- row fidelity ----------
def compute_pairwise_row_fidelity(real_df: pd.DataFrame, synth_df: pd.DataFrame,
                                  categorical_cols: List[str], numeric_cols: List[str]) -> Tuple[Dict, float]:
    per_pair = {}
    scores = []

    # numeric-numeric pairs
    for a, b in combinations(numeric_cols, 2):
        ra = real_df[a].dropna().values
        rb = real_df[b].dropna().values
        sa = synth_df[a].dropna().values
        sb = synth_df[b].dropna().values
        try:
            rho_real = float(pearsonr(ra, rb)[0])
        except Exception:
            rho_real = 0.0
        try:
            rho_synth = float(pearsonr(sa, sb)[0])
        except Exception:
            rho_synth = 0.0
        delta = abs(rho_real - rho_synth)
        omega = 1.0 - 0.5 * delta
        per_pair[(a,b)] = {"type": "num-num", "rho_real": rho_real, "rho_synth": rho_synth, "omega_row": float(omega)}
        scores.append(omega)

    # categorical-categorical pairs (joint TVD)
    for a, b in combinations(categorical_cols, 2):
        ra = real_df[[a,b]].astype(str).fillna("__nan__")
        sa = synth_df[[a,b]].astype(str).fillna("__nan__")
        real_joint = ra.value_counts().sort_index()
        synth_joint = sa.value_counts().reindex(real_joint.index, fill_value=0)
        tvd = total_variation_distance_counts(real_joint.values, synth_joint.values)
        omega = 1.0 - 0.5 * float(tvd)
        per_pair[(a,b)] = {"type": "cat-cat", "tvd_joint": float(tvd), "omega_row": float(omega)}
        scores.append(omega)

    Omega_row = float(np.mean(scores)) if len(scores) > 0 else 0.0
    return per_pair, Omega_row

# ---------- DCR (privacy) ----------
def compute_DCR(real_df: pd.DataFrame, synth_df: pd.DataFrame,
                categorical_cols: List[str], numeric_cols: List[str]) -> float:
    scaler = StandardScaler()
    real_nums = real_df[numeric_cols].astype(float).fillna(0.0).values if len(numeric_cols)>0 else np.zeros((len(real_df),0))
    synth_nums = synth_df[numeric_cols].astype(float).fillna(0.0).values if len(numeric_cols)>0 else np.zeros((len(synth_df),0))
    if real_nums.shape[1] > 0:
        scaler.fit(real_nums)
        real_nums_s = scaler.transform(real_nums)
        synth_nums_s = scaler.transform(synth_nums)
    else:
        real_nums_s = np.zeros((len(real_df),0))
        synth_nums_s = np.zeros((len(synth_df),0))

    enc = OneHotEncoder(handle_unknown="ignore", sparse=False)
    if len(categorical_cols) > 0:
        real_cats = real_df[categorical_cols].astype(str).fillna("__nan__").values
        synth_cats = synth_df[categorical_cols].astype(str).fillna("__nan__").values
        enc.fit(real_cats)
        real_cats_ohe = enc.transform(real_cats)
        synth_cats_ohe = enc.transform(synth_cats)
    else:
        real_cats_ohe = np.zeros((len(real_df),0))
        synth_cats_ohe = np.zeros((len(synth_df),0))

    real_matrix = np.hstack([real_nums_s, real_cats_ohe])
    synth_matrix = np.hstack([synth_nums_s, synth_cats_ohe])

    nn = NearestNeighbors(n_neighbors=1, algorithm='auto', metric='euclidean')
    nn.fit(real_matrix)
    dists, _ = nn.kneighbors(synth_matrix, n_neighbors=1, return_distance=True)
    median_dcr = float(np.median(dists.ravel()))
    return median_dcr

# ---------- synthesis (exact match) ----------
def compute_synthesis(real_df: pd.DataFrame, synth_df: pd.DataFrame,
                      categorical_cols: List[str], numeric_cols: List[str]) -> float:
    if len(categorical_cols) > 0:
        real_cat_keys = real_df[categorical_cols].astype(str).fillna("__nan__").agg("||".join, axis=1)
        synth_cat_keys = synth_df[categorical_cols].astype(str).fillna("__nan__").agg("||".join, axis=1)
        groups = {}
        for idx, key in enumerate(real_cat_keys):
            groups.setdefault(key, []).append(idx)
    else:
        real_cat_keys = pd.Series(["_all_"] * len(real_df))
        synth_cat_keys = pd.Series(["_all_"] * len(synth_df))
        groups = {"_all_": list(range(len(real_df)))}

    real_nums = real_df[numeric_cols].astype(float).fillna(0.0).values
    synth_nums = synth_df[numeric_cols].astype(float).fillna(0.0).values

    matched = 0
    for i, key in enumerate(synth_cat_keys):
        candidate_idxs = groups.get(key, [])
        if len(candidate_idxs) == 0:
            continue
        s_vals = synth_nums[i] if len(numeric_cols)>0 else np.array([])
        r_vals = real_nums[candidate_idxs] if len(numeric_cols)>0 else np.array([[]])
        if r_vals.size == 0 and s_vals.size == 0:
            matched += 1
            continue
        abs_diffs = np.abs(r_vals - s_vals[None, :])
        rel_bounds = 0.01 * (np.abs(r_vals) + 1e-12)
        within = (abs_diffs <= rel_bounds) | (abs_diffs <= 1e-6)
        row_all = within.all(axis=1)
        if row_all.any():
            matched += 1
    return float(matched / len(synth_df))

# # ---------- utility (train on synth, test on real) ----------
# def compute_utility_accuracy(real_test_df: pd.DataFrame, synth_train_df: pd.DataFrame,
#                              label_col: str) -> Tuple[Dict[str,float], float]:

#     models = {
#         # 🌲 Random Forest — deeper trees, balanced subsampling for stability
#         "RandomForest": RandomForestClassifier(
#             n_estimators=300,           # more trees → better generalization
#             max_depth=15,               # limits overfitting
#             min_samples_split=4,        # ensures splits are meaningful
#             min_samples_leaf=2,
#             max_features="sqrt",        # common best practice for classification
#             bootstrap=True,
#             n_jobs=-1,
#             random_state=42
#         ),
    
#         # 🌳 Decision Tree — slightly regularized to avoid overfitting
#         "DecisionTree": DecisionTreeClassifier(
#             criterion="gini",
#             max_depth=10,               # limits overfitting
#             min_samples_split=5,
#             min_samples_leaf=3,
#             random_state=42
#         ),
    
#         # 📈 Logistic Regression — stronger regularization and robust solver
#         "LogisticRegression": LogisticRegression(
#             max_iter=1000,
#             solver="lbfgs"
#         ),
    
#         # 🚀 AdaBoost — more estimators + tuned learning rate
#         "AdaBoost": AdaBoostClassifier(
#             n_estimators=200,
#             learning_rate=0.5,          # smaller LR helps smoother convergence
#             random_state=42
#         )
#             #,
    
#         # # 🤓 Naive Bayes — standard (no major tuning options)
#         # "NaiveBayes": GaussianNB(var_smoothing=1e-9)  # small var smoothing helps numerical stability
#     }

#     synth_train_df = synth_train_df[real_test_df.columns]
#     X_train=synth_train_df.drop(columns=[label_col])
#     X_test=real_test_df.drop(columns=[label_col])
#     y_train=synth_train_df[label_col]
#     y_test= real_test_df[label_col]

#     accuracies = {}
#     accs=[]
#     for name, model in models.items():
#         model.fit(X_train, y_train)
#         preds = model.predict(X_test)
#         curr_acc=accuracy_score(y_test, preds)
#         accuracies[name] = curr_acc
#         accs.append(curr_acc)
#     mean_acc = float(np.mean(accs))
#     return accuracies, mean_acc

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

def compute_utility_accuracy(real_test_df: pd.DataFrame,
                             synth_train_df: pd.DataFrame,
                             label_col: str):

    models = {
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            max_depth=15,
            min_samples_split=4,
            min_samples_leaf=2,
            max_features="sqrt",
            bootstrap=True,
            n_jobs=-1,
            random_state=42
        ),
        "DecisionTree": DecisionTreeClassifier(
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=3,
            random_state=42
        ),
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            solver="lbfgs"
        ),
        "AdaBoost": AdaBoostClassifier(
            n_estimators=200,
            learning_rate=0.5,
            random_state=42
        )
    }

    # Align columns
    synth_train_df = synth_train_df[real_test_df.columns]

    X_train = synth_train_df.drop(columns=[label_col])
    y_train = synth_train_df[label_col]

    X_test = real_test_df.drop(columns=[label_col])
    y_test = real_test_df[label_col]

    # Identify categorical & numeric columns automatically
    categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    numeric_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

    # Preprocessing pipeline
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", "passthrough", numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
        ]
    )

    accuracies = []
    results = {}

    for name, model in models.items():
        clf = Pipeline(steps=[
            ("preprocess", preprocessor),
            ("model", model)
        ])

        clf.fit(X_train, y_train)
        preds = clf.predict(X_test)
        acc = accuracy_score(y_test, preds)

        results[name] = acc
        accuracies.append(acc)

    return results, float(np.mean(accuracies))


In [2]:
numeric_cols=['age', 'fnlwgt', 'educational-num', 'capital-gain', 'capital-loss','hours-per-week']
categorical_cols=['workclass', 'education', 'marital-status', 'occupation','relationship', 'race', 'gender', 'native-country', 'income']

In [5]:
label_col = "income"  
real_df = pd.read_csv("/kaggle/input/datasets/toshangupta/adult-dataset/adult.csv")
synth_df = pd.read_csv("/kaggle/input/datasets/toshangupta/synthetic-adult-data-findiff/synthetic.csv")
cat_cols = categorical_cols
num_cols = numeric_cols

print("Computing column fidelity...")
per_col, Omega_col = column_fidelity(real_df, synth_df, cat_cols, num_cols)

print("Computing row fidelity (pairwise)...")
per_pair, Omega_row = compute_pairwise_row_fidelity(real_df, synth_df, cat_cols, num_cols)

print("Computing privacy (DCR median)...")
median_dcr = compute_DCR(real_df, synth_df, cat_cols, num_cols)

print("Computing synthesis fraction (exact/1% matches)...")
synth_frac = compute_synthesis(real_df, synth_df, cat_cols, num_cols)

print("Computing utility (train on synth, test on real)...")
utility_per_clf, utility_mean = compute_utility_accuracy(real_df, synth_df,label_col)

# Summarize
summary = {
    "Omega_col": Omega_col,
    "Omega_row": Omega_row,
    "privacy_median_DCR": median_dcr,
    "synthesis_fraction": synth_frac,
    "utility_mean_accuracy": utility_mean,
    "utility_per_classifier": utility_per_clf
}

print("\n=== Evaluation Summary ===")
print(f"Omega_col (column fidelity avg): {Omega_col:.4f}")
print(f"Omega_row (row fidelity avg):    {Omega_row:.4f}")
print(f"Privacy (median DCR):            {median_dcr:.6f}")
print(f"Utility (mean accuracy):         {utility_mean:.4f}")
print(f"Synthesis (match frac):          {synth_frac:.6f}")

Computing column fidelity...
Computing row fidelity (pairwise)...
Computing privacy (DCR median)...


/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Computing synthesis fraction (exact/1% matches)...
Computing utility (train on synth, test on real)...

=== Evaluation Summary ===
Omega_col (column fidelity avg): 0.9620
Omega_row (row fidelity avg):    0.9377
Privacy (median DCR):            0.955990
Utility (mean accuracy):         0.8380
Synthesis (match frac):          0.006311


In [6]:
# Cell 3: save results to disk for record-keeping
out_json = "finDiff_eval_results.json"
out_csv = "finDiff_eval_per_column.csv"

# write JSON
with open(out_json, "w") as f:
    json.dump({
        "summary": summary,
        "per_column": per_col,
        "per_pair": {str(k):v for k,v in per_pair.items()}
    }, f, indent=2)
print("Saved evaluation JSON:", out_json)

# optional: save per-column table
per_col_df = pd.DataFrame.from_dict(per_col, orient="index")
per_col_df.to_csv(out_csv)
print("Saved per-column CSV:", out_csv)

Saved evaluation JSON: finDiff_eval_results.json
Saved per-column CSV: finDiff_eval_per_column.csv
